#### 20 PERCENT OVERSAMPLED 

In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from collections import Counter
import os

# Load original reduced dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# Define target and feature columns
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# Feature engineering
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Custom ADASYN resampling for 20% positive class ratio
def adasyn_resample_custom(X, y, target_ratio=0.20):
    X_res_list = []
    y_res_list = []

    for target in targets:
        print(f"\nResampling target: {target}")
        counts = Counter(y[target])
        majority_class = counts[0]
        desired_minority = int(majority_class * target_ratio / (1 - target_ratio))
        print(f"Original counts: {counts}, Desired minority: {desired_minority}")

        # Perform ADASYN oversampling
        sampler = ADASYN(sampling_strategy={1: desired_minority}, random_state=42, n_neighbors=5)
        X_res, y_res = sampler.fit_resample(X, y[target])

        X_res = pd.DataFrame(X_res, columns=X.columns)
        y_res = pd.Series(y_res, name=target)

        X_res_list.append(X_res)
        y_res_list.append(y_res)

    return X_res_list, y_res_list

# Run oversampling
X_resampled_list, y_resampled_list = adasyn_resample_custom(X_train, y_train, target_ratio=0.20)

# Save output CSVs
os.makedirs("adasyn_20pct", exist_ok=True)
for i, target in enumerate(targets):
    df_combined = pd.concat([X_resampled_list[i], y_resampled_list[i]], axis=1)
    df_combined.to_csv(f"adasyn_20pct/adasyn_20pct_{target}.csv", index=False)
    print(f"Saved: adasyn_20pct/adasyn_20pct_{target}.csv | Shape: {df_combined.shape}")



Resampling target: oestrus
Original counts: Counter({0: 253490, 1: 1042}), Desired minority: 63372

Resampling target: calving
Original counts: Counter({0: 253961, 1: 571}), Desired minority: 63490

Resampling target: lameness
Original counts: Counter({0: 254115, 1: 417}), Desired minority: 63528

Resampling target: mastitis
Original counts: Counter({0: 254406, 1: 126}), Desired minority: 63601
Saved: adasyn_20pct/adasyn_20pct_oestrus.csv | Shape: (317276, 9)
Saved: adasyn_20pct/adasyn_20pct_calving.csv | Shape: (317612, 9)
Saved: adasyn_20pct/adasyn_20pct_lameness.csv | Shape: (317699, 9)
Saved: adasyn_20pct/adasyn_20pct_mastitis.csv | Shape: (318010, 9)


#### RANDOM FOREST AND LIGHT GBM APPLIED

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import classification_report, make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import joblib
import os

# 1. Configuration
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
data_dir = "adasyn_20pct"  # Folder for 20% ADASYN
model_dir = "models/20pct"
os.makedirs(model_dir, exist_ok=True)

# 2. Scoring and Cross-validation setup
scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted"))
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# 3. Model training function
def tune_and_train(X_train, y_train, model_type):
    if model_type == "rf":
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [10, 20],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }
    else:  # LightGBM
        base_model = LGBMClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [200, 300],
            "max_depth": [12, 16],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "class_weight": ["balanced"]
        }

    grid = GridSearchCV(
        base_model,
        param_grid,
        scoring=scorer,
        cv=cv,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best params for {model_type.upper()}: {grid.best_params_}")
    return grid.best_estimator_

# 4. Train both RF and LGBM
for model_type in ["rf", "lgbm"]:
    print(f"\n=== Training {model_type.upper()} models with 20% ADASYN oversampling ===\n")

    for target in targets:
        print(f"--- {target} ---")
        # Load dataset
        df_res = pd.read_csv(f"{data_dir}/adasyn_20pct_{target}.csv")

        X = df_res.drop(columns=[target])
        y = df_res[target]

        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        best_model = tune_and_train(X_train, y_train, model_type)

        y_pred = best_model.predict(X_test)
        print(f"\nClassification Report for {model_type.upper()} - {target}:")
        print(classification_report(y_test, y_pred))

        joblib.dump(best_model, f"{model_dir}/{model_type}_{target}.pkl")
        print(f"Saved model at {model_dir}/{model_type}_{target}.pkl")



=== Training RF models with 20% ADASYN oversampling ===

--- oestrus ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - oestrus:
              precision    recall  f1-score   support

           0       0.97      0.95      0.96     50699
           1       0.82      0.88      0.85     12757

    accuracy                           0.94     63456
   macro avg       0.89      0.92      0.90     63456
weighted avg       0.94      0.94      0.94     63456

Saved model at models/20pct/rf_oestrus.pkl
--- calving ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - calving:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     50793
       

#### MLP APPLIED

In [21]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ["oestrus", "calving", "lameness", "mastitis"]
model_dir = "models/20pct_mlp"
os.makedirs(model_dir, exist_ok=True)

# MLP architecture
class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

# Training function
def train_mlp(X, y, target_name):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=42
    )

    train_data = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train.values, dtype=torch.float32)
    )
    test_data = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test.values, dtype=torch.float32)
    )

    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=64)

    model = MLPModel(input_dim=X.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(10):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                out = torch.sigmoid(model(xb)).squeeze()
                preds_all.extend((out > 0.5).int().cpu().numpy())
                labels_all.extend(yb.int().cpu().numpy())

        print(f"\nEpoch {epoch+1} - {target_name}:\n")
        print(classification_report(labels_all, preds_all))

    save_path = os.path.join(model_dir, f"mlp_{target_name}_20pct.pth")
    torch.save(model.state_dict(), save_path)
    print(f"Saved model to: {save_path}")

# Main loop
for target in TARGETS:
    df = pd.read_csv(f"adasyn_20pct/adasyn_20pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]
    train_mlp(X, y, target)



Epoch 1 - oestrus:

              precision    recall  f1-score   support

           0       0.87      0.99      0.93     50699
           1       0.95      0.42      0.58     12757

    accuracy                           0.88     63456
   macro avg       0.91      0.71      0.76     63456
weighted avg       0.89      0.88      0.86     63456


Epoch 2 - oestrus:

              precision    recall  f1-score   support

           0       0.88      0.99      0.93     50699
           1       0.93      0.48      0.63     12757

    accuracy                           0.89     63456
   macro avg       0.90      0.74      0.78     63456
weighted avg       0.89      0.89      0.87     63456


Epoch 3 - oestrus:

              precision    recall  f1-score   support

           0       0.89      0.99      0.94     50699
           1       0.91      0.52      0.66     12757

    accuracy                           0.89     63456
   macro avg       0.90      0.75      0.80     63456
weighted av

#### TABNET APPLIED

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
import os

# ----------------------
# Setup
# ----------------------
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
data_dir = "adasyn_20pct"
model_dir = "models/20pct_tabnet"
os.makedirs(model_dir, exist_ok=True)

# ----------------------
# Training Loop
# ----------------------
for target in targets:
    print(f"\n=== Training TabNet for target: {target} (20% ADASYN) ===")
    
    # Load data
    df = pd.read_csv(f"{data_dir}/adasyn_20pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Encode target
    y_train_enc = LabelEncoder().fit_transform(y_train)
    y_test_enc = LabelEncoder().fit_transform(y_test)

    # Define TabNet model
    clf = TabNetClassifier(
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-2),
        scheduler_params={"step_size":10, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        verbose=0,
        seed=42
    )

    # Train model
    clf.fit(
        X_train=X_train_scaled, y_train=y_train_enc,
        eval_set=[(X_test_scaled, y_test_enc)],
        eval_name=["val"],
        eval_metric=["accuracy"],
        max_epochs=100,
        patience=10,
        batch_size=1024,
        virtual_batch_size=128
    )

    # Predict and evaluate
    preds = clf.predict(X_test_scaled)
    report = classification_report(y_test_enc, preds, target_names=['0', '1'])
    print(report)

    # Save model
    clf.save_model(f"{model_dir}/tabnet_{target}_20pct")



=== Training TabNet for target: oestrus (20% ADASYN) ===

Early stopping occurred at epoch 17 with best_epoch = 7 and best_val_accuracy = 0.89591


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.90      0.99      0.94     50737
           1       0.90      0.54      0.67     12719

    accuracy                           0.90     63456
   macro avg       0.90      0.76      0.81     63456
weighted avg       0.90      0.90      0.89     63456

Successfully saved model at models/20pct_tabnet/tabnet_oestrus_20pct.zip

=== Training TabNet for target: calving (20% ADASYN) ===

Early stopping occurred at epoch 20 with best_epoch = 10 and best_val_accuracy = 0.93563


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.93      0.99      0.96     50870
           1       0.94      0.72      0.82     12653

    accuracy                           0.94     63523
   macro avg       0.94      0.86      0.89     63523
weighted avg       0.94      0.94      0.93     63523

Successfully saved model at models/20pct_tabnet/tabnet_calving_20pct.zip

=== Training TabNet for target: lameness (20% ADASYN) ===

Early stopping occurred at epoch 25 with best_epoch = 15 and best_val_accuracy = 0.89932


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.89      1.00      0.94     50922
           1       0.96      0.51      0.67     12618

    accuracy                           0.90     63540
   macro avg       0.93      0.75      0.80     63540
weighted avg       0.91      0.90      0.89     63540

Successfully saved model at models/20pct_tabnet/tabnet_lameness_20pct.zip

=== Training TabNet for target: mastitis (20% ADASYN) ===

Early stopping occurred at epoch 13 with best_epoch = 3 and best_val_accuracy = 0.894


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.90      0.98      0.94     50883
           1       0.89      0.54      0.67     12719

    accuracy                           0.89     63602
   macro avg       0.89      0.76      0.80     63602
weighted avg       0.89      0.89      0.88     63602

Successfully saved model at models/20pct_tabnet/tabnet_mastitis_20pct.zip


#### LSTM APPLIED

In [15]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

# -------------------------
# Define LSTM Classifier
# -------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # Take output from the last time step
        out = self.fc(out)
        return self.sigmoid(out)

# -------------------------
# Train Function
# -------------------------
def train_lstm_model(X, y, target_name, model_save_path):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.expand_dims(X_scaled, axis=1)  # Reshape to [batch, seq_len=1, features]

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=42
    )

    train_data = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train.values, dtype=torch.float32))
    test_data = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.float32))

    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=64)

    model = LSTMClassifier(input_dim=X_train.shape[2], hidden_dim=64, output_dim=1)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):  # You can increase if needed
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # Evaluation
        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                out = model(xb).squeeze()
                preds_all.extend((out > 0.5).int().cpu().numpy())
                labels_all.extend(yb.int().cpu().numpy())

        print(f"\nEpoch {epoch+1} - {target_name}:\n")
        print(classification_report(labels_all, preds_all))

    # Save model
    model_path = os.path.join(model_save_path, f"lstm_{target_name}_20pct.pth")
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to: {model_path}\n")
    return model_path

# -------------------------
# Run for All Targets
# -------------------------
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
os.makedirs("models/20pct_lstm", exist_ok=True)

for target in targets:
    df = pd.read_csv(f"adasyn_20pct/adasyn_20pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]
    train_lstm_model(X, y, target, model_save_path="models/20pct_lstm")



Epoch 1 - oestrus:

              precision    recall  f1-score   support

           0       0.88      0.98      0.93     50699
           1       0.86      0.49      0.63     12757

    accuracy                           0.88     63456
   macro avg       0.87      0.74      0.78     63456
weighted avg       0.88      0.88      0.87     63456


Epoch 2 - oestrus:

              precision    recall  f1-score   support

           0       0.89      0.98      0.93     50699
           1       0.89      0.51      0.65     12757

    accuracy                           0.89     63456
   macro avg       0.89      0.75      0.79     63456
weighted avg       0.89      0.89      0.88     63456


Epoch 3 - oestrus:

              precision    recall  f1-score   support

           0       0.89      0.98      0.94     50699
           1       0.90      0.52      0.66     12757

    accuracy                           0.89     63456
   macro avg       0.89      0.75      0.80     63456
weighted av

#### TCN APPLIED

In [17]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.optim import Adam
from torch.nn.utils import weight_norm
import random

# Fix seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Dataset class
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Chomp1d block
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

# Temporal block
class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride,
                                           padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride,
                                           padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)

        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

# TCN model
class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=3, dropout=0.3):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i - 1]
            out_channels = num_channels[i]
            layers.append(
                TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                              dilation=dilation_size, padding=(kernel_size - 1) * dilation_size,
                              dropout=dropout)
            )
        self.network = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        y1 = self.network(x)
        y1 = y1[:, :, -1]  # Last time step
        return self.linear(y1)

# TCN training function
def train_tcn_model(target):
    data_path = f'adasyn_20pct/adasyn_20pct_{target}.csv'
    df = pd.read_csv(data_path)
    features = df.drop(columns=[target])
    labels = df[target]

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Reshape to [samples, channels, sequence_length]
    X = features_scaled.reshape(features_scaled.shape[0], features_scaled.shape[1], 1)
    X = np.transpose(X, (0, 2, 1))  # [batch, channels, time]

    X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42, stratify=labels)

    train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=256, shuffle=True)
    test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=256, shuffle=False)

    model = TCN(input_size=1, output_size=2, num_channels=[16, 32, 64])
    model = model.cuda() if torch.cuda.is_available() else model
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)

    best_accuracy = 0
    for epoch in range(1, 11):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            if torch.cuda.is_available():
                X_batch, y_batch = X_batch.cuda(), y_batch.cuda()
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Evaluate
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                if torch.cuda.is_available():
                    X_batch = X_batch.cuda()
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                y_true.extend(y_batch.numpy())
                y_pred.extend(predicted.cpu().numpy())

        print(f"\nEpoch {epoch} - {target}:\n")
        print(classification_report(y_true, y_pred, digits=4))

        acc = (np.array(y_pred) == np.array(y_true)).mean()
        if acc > best_accuracy:
            best_accuracy = acc
            save_path = f'models/20pct_tcn/tcn_{target}_20pct.pth'
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to: {save_path}")

# Train TCN for all diseases
for target_name in ['oestrus', 'calving', 'lameness', 'mastitis']:
    train_tcn_model(target_name)


C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - oestrus:

              precision    recall  f1-score   support

           0     0.8652    0.9932    0.9248     50699
           1     0.9343    0.3849    0.5452     12757

    accuracy                         0.8709     63456
   macro avg     0.8998    0.6890    0.7350     63456
weighted avg     0.8791    0.8709    0.8485     63456

Model saved to: models/20pct_tcn/tcn_oestrus_20pct.pth

Epoch 2 - oestrus:

              precision    recall  f1-score   support

           0     0.8860    0.9832    0.9321     50699
           1     0.8817    0.4972    0.6359     12757

    accuracy                         0.8855     63456
   macro avg     0.8839    0.7402    0.7840     63456
weighted avg     0.8851    0.8855    0.8725     63456

Model saved to: models/20pct_tcn/tcn_oestrus_20pct.pth

Epoch 3 - oestrus:

              precision    recall  f1-score   support

           0     0.8874    0.9884    0.9352     50699
           1     0.9161    0.5016    0.6483     12757

    accur

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - calving:

              precision    recall  f1-score   support

           0     0.9241    0.9845    0.9533     50793
           1     0.9164    0.6774    0.7790     12730

    accuracy                         0.9230     63523
   macro avg     0.9202    0.8309    0.8662     63523
weighted avg     0.9226    0.9230    0.9184     63523

Model saved to: models/20pct_tcn/tcn_calving_20pct.pth

Epoch 2 - calving:

              precision    recall  f1-score   support

           0     0.9289    0.9842    0.9558     50793
           1     0.9174    0.6994    0.7937     12730

    accuracy                         0.9271     63523
   macro avg     0.9231    0.8418    0.8747     63523
weighted avg     0.9266    0.9271    0.9233     63523

Model saved to: models/20pct_tcn/tcn_calving_20pct.pth

Epoch 3 - calving:

              precision    recall  f1-score   support

           0     0.9325    0.9843    0.9577     50793
           1     0.9194    0.7156    0.8048     12730

    accur

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - lameness:

              precision    recall  f1-score   support

           0     0.8783    0.9937    0.9324     50823
           1     0.9470    0.4495    0.6096     12717

    accuracy                         0.8848     63540
   macro avg     0.9126    0.7216    0.7710     63540
weighted avg     0.8920    0.8848    0.8678     63540

Model saved to: models/20pct_tcn/tcn_lameness_20pct.pth

Epoch 2 - lameness:

              precision    recall  f1-score   support

           0     0.8895    0.9868    0.9356     50823
           1     0.9062    0.5100    0.6527     12717

    accuracy                         0.8914     63540
   macro avg     0.8979    0.7484    0.7942     63540
weighted avg     0.8928    0.8914    0.8790     63540

Model saved to: models/20pct_tcn/tcn_lameness_20pct.pth

Epoch 3 - lameness:

              precision    recall  f1-score   support

           0     0.8880    0.9908    0.9366     50823
           1     0.9315    0.5004    0.6511     12717

    

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - mastitis:

              precision    recall  f1-score   support

           0     0.8855    0.9796    0.9302     50881
           1     0.8581    0.4933    0.6264     12721

    accuracy                         0.8823     63602
   macro avg     0.8718    0.7364    0.7783     63602
weighted avg     0.8800    0.8823    0.8694     63602

Model saved to: models/20pct_tcn/tcn_mastitis_20pct.pth

Epoch 2 - mastitis:

              precision    recall  f1-score   support

           0     0.8844    0.9856    0.9323     50881
           1     0.8938    0.4849    0.6288     12721

    accuracy                         0.8855     63602
   macro avg     0.8891    0.7353    0.7805     63602
weighted avg     0.8863    0.8855    0.8716     63602

Model saved to: models/20pct_tcn/tcn_mastitis_20pct.pth

Epoch 3 - mastitis:

              precision    recall  f1-score   support

           0     0.8926    0.9717    0.9305     50881
           1     0.8246    0.5324    0.6471     12721

    

#### HYBRID ENSEMBLE METHOD

In [23]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib
from pytorch_tabnet.tab_model import TabNetClassifier
import pickle

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ["oestrus", "calving", "lameness", "mastitis"]
FEATURES_8 = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL','hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio']

model_dirs = {
    "rf": "models/20pct",
    "lgbm": "models/20pct",
    "lstm": "models/20pct_lstm",
    "tcn": "models/20pct_tcn",
    "tabnet": "models/20pct_tabnet",
    "mlp": "models/20pct_mlp",
    "ensemble": "models/20pct_ensemble"
}

# LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# TCN Model
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                                    stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.utils.weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                                    stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_size=1, output_size=2, num_channels=[16, 32, 64], kernel_size=3, dropout=0.3):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                        dilation=dilation_size, padding=(kernel_size-1)*dilation_size,
                                        dropout=dropout))
        self.network = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        x = self.network(x)
        return self.linear(x[:, :, -1])  # Last time step

# MLP model
class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

# Inference loop
for target in TARGETS:
    print(f"\n--- Hybrid Ensemble (20% ADASYN) for: {target} ---")

    df = pd.read_csv(f"adasyn_20pct/adasyn_20pct_{target}.csv")
    X = df[FEATURES_8]
    y = df[target]

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

    rf = joblib.load(os.path.join(model_dirs['rf'], f"rf_{target}.pkl"))
    lgbm = joblib.load(os.path.join(model_dirs['lgbm'], f"lgbm_{target}.pkl"))
    tabnet = TabNetClassifier()
    tabnet.load_model(os.path.join(model_dirs['tabnet'], f"tabnet_{target}_20pct.zip"))

    rf_probs = rf.predict_proba(X_test)[:, 1]
    lgbm_probs = lgbm.predict_proba(X_test)[:, 1]
    tabnet_probs = tabnet.predict_proba(X_test.values)[:, 1]

    # LSTM
    X_seq = X.values.reshape(-1, 1, X.shape[1])
    X_seq_tensor = torch.tensor(X_seq, dtype=torch.float32).to(DEVICE)
    lstm_model = LSTMModel(input_size=X.shape[1])
    lstm_path = os.path.join(model_dirs['lstm'], f"lstm_{target}_20pct.pth")
    lstm_model.load_state_dict(torch.load(lstm_path, map_location=DEVICE))
    lstm_model.to(DEVICE).eval()
    with torch.no_grad():
        lstm_logits = lstm_model(X_seq_tensor)
        lstm_probs = torch.sigmoid(lstm_logits).cpu().numpy().squeeze()

    # TCN
    tcn_model = TCNModel()
    tcn_path = os.path.join(model_dirs['tcn'], f"tcn_{target}_20pct.pth")
    tcn_model.load_state_dict(torch.load(tcn_path, map_location=DEVICE))
    tcn_model.to(DEVICE).eval()
    with torch.no_grad():
        tcn_probs = tcn_model(X_seq_tensor).softmax(dim=1).cpu().numpy()[:, 1]

    # MLP
    mlp_model = MLPModel(input_dim=X.shape[1])
    mlp_path = os.path.join(model_dirs['mlp'], f"mlp_{target}_20pct.pth")
    mlp_model.load_state_dict(torch.load(mlp_path, map_location=DEVICE))
    mlp_model.to(DEVICE).eval()
    with torch.no_grad():
        mlp_probs = torch.sigmoid(mlp_model(torch.tensor(X_test.values, dtype=torch.float32).to(DEVICE))).cpu().numpy().squeeze()

    # Average all probabilities
    min_len = min(len(rf_probs), len(lgbm_probs), len(tabnet_probs), len(lstm_probs), len(tcn_probs), len(mlp_probs))
    final_prob = (
        rf_probs[:min_len] +
        lgbm_probs[:min_len] +
        tabnet_probs[:min_len] +
        lstm_probs[:min_len] +
        tcn_probs[:min_len] +
        mlp_probs[:min_len]
    ) / 6

    final_preds = (final_prob > 0.5).astype(int)
    y_true = y_test[:min_len]

    print(classification_report(y_true, final_preds, digits=4))

    # Save ensemble predictions
    os.makedirs(model_dirs['ensemble'], exist_ok=True)
    output_path = os.path.join(model_dirs['ensemble'], f"ensemble_preds_{target}.pkl")
    with open(output_path, "wb") as f:
        pickle.dump({
            "y_true": y_true,
            "y_pred": final_preds,
            "y_prob": final_prob
        }, f)
    print(f"Saved ensemble predictions for {target} to: {output_path}")



--- Hybrid Ensemble (20% ADASYN) for: oestrus ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.8096    0.9982    0.8940     50699
           1     0.9026    0.0669    0.1245     12757

    accuracy                         0.8110     63456
   macro avg     0.8561    0.5325    0.5093     63456
weighted avg     0.8283    0.8110    0.7393     63456

Saved ensemble predictions for oestrus to: models/20pct_ensemble\ensemble_preds_oestrus.pkl

--- Hybrid Ensemble (20% ADASYN) for: calving ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.8654    0.9600    0.9102     50793
           1     0.7170    0.4042    0.5169     12730

    accuracy                         0.8486     63523
   macro avg     0.7912    0.6821    0.7136     63523
weighted avg     0.8356    0.8486    0.8314     63523

Saved ensemble predictions for calving to: models/20pct_ensemble\ensemble_preds_calving.pkl

--- Hybrid Ensemble (20% ADASYN) for: lameness ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.8022    0.9999    0.8902     50823
           1     0.9734    0.0144    0.0284     12717

    accuracy                         0.8027     63540
   macro avg     0.8878    0.5071    0.4593     63540
weighted avg     0.8364    0.8027    0.7177     63540

Saved ensemble predictions for lameness to: models/20pct_ensemble\ensemble_preds_lameness.pkl

--- Hybrid Ensemble (20% ADASYN) for: mastitis ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.8951    0.9313    0.9128     50881
           1     0.6721    0.5632    0.6129     12721

    accuracy                         0.8577     63602
   macro avg     0.7836    0.7473    0.7628     63602
weighted avg     0.8505    0.8577    0.8528     63602

Saved ensemble predictions for mastitis to: models/20pct_ensemble\ensemble_preds_mastitis.pkl
